# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-awaischohan/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane 4: CTR / Engagement Opportunity Scoring. Task type: ranking, built on a score.**

Each page gets a score for how far its click-through rate (CTR) sits below what pages at a similar position normally get. The score is then sorted into a review queue. I call it ranking, not classification, because nobody acts on a yes/no flag. A reviewer has time for a fixed number of pages (I assume 50 per sprint for now) and needs to know which ones to open first. Clustering does not fit: I am not looking for page types, I am ordering pages by priority.

**The loop, for this lane:**
- **World:** pages that already rank on page 1 or 2 but get fewer clicks than similar pages.
- **Data:** the 90-day starter slice, plus its two 30-day windows.
- **Output:** a ranked list with a position-adjusted CTR gap score and reason codes.
- **Decision and action:** a reviewer picks which pages to open, then checks the title, meta description and intent match, or marks the page "monitor".
- **New data:** the pages' CTR in later windows. This closes the loop and is how I can check whether the ranking was any good.

In [ ]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/m-awaischohan/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Lane slice: visible pages on page 1-2 with enough volume in BOTH 30-day windows
lane = df[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0) & (df["avg_position"] <= 20)          # 0 means "no data"
    & df["position_tier"].isin(["top_3", "page_1", "striking"])
    & (df["impressions_prev_30d"] >= 100)
    & (df["impressions_last_30d"] >= 100)
].copy()

print(f"Full starter file: {len(df):,} pages")
print(f"Lane slice:        {len(lane):,} pages  ({len(lane)/len(df):.0%} of the file)")
print(f"Clients in slice:  {lane['client_id'].nunique()}")
print("\nThe output to rank is therefore ~", f"{len(lane):,}", "pages -> a top-50 queue is the thing a reviewer sees.")

Full starter file: 30,000 pages
Lane slice:        10,835 pages  (36% of the file)
Clients in slice:  28

The output to rank is therefore ~ 10,835 pages -> a top-50 queue is the thing a reviewer sees.


## 2. Target or proxy

I need to be upfront: **in this slice there is no observed "this page was fixed and clicks improved" outcome.** Whatever I choose is a proxy, and I will say so everywhere I use it.

**Target I will sketch: `low_last` = 1 when a page's CTR in the most recent 30 days is below half of what its position tier normally gets in that same window.**
- *Feature window:* the earlier 30 days (`*_prev_30d`) plus page properties that do not change.
- *Target window:* the most recent 30 days (`*_last_30d`).
- *Why this one:* it is a later-window outcome, not a rule applied to the same numbers I use as inputs. The model has to find pages whose gap is likely to **persist**, not just re-compute today's gap.

**What is observed and what is defined:** the clicks and impressions are observed. The "half of tier typical" cut-off, and the tier benchmarks, are my own definitions. A different cut-off gives a different label, so I will test alternatives later.

**Known weaknesses I already see (to be fixed in the data contract, week 3):**
- `avg_position` is a 90-day average, so it overlaps the target window. That is a mild leak until I can get per-window position from the warehouse.
- Columns like `ctr`, `clicks_90d` and `impressions_90d` include the target window, so they cannot be features for this target.
- The tier benchmark is computed from the same slice, so it is slightly circular. Using the warehouse for benchmarks fixes this.

In [ ]:
# Sketch of the target column
for w in ("prev", "last"):
    lane[f"ctr_{w}"] = 100 * lane[f"clicks_{w}_30d"] / lane[f"impressions_{w}_30d"]   # already in % units
    # typical CTR per position tier in that window = pooled clicks / pooled impressions
    sums = lane.groupby("position_tier")[[f"clicks_{w}_30d", f"impressions_{w}_30d"]].sum()
    bench = 100 * sums[f"clicks_{w}_30d"] / sums[f"impressions_{w}_30d"]
    lane[f"expected_ctr_{w}"] = lane["position_tier"].map(bench)
    print(f"Typical CTR by tier, {w} 30 days:", bench.round(2).to_dict())

lane["gap_prev"] = lane["ctr_prev"] - lane["expected_ctr_prev"]            # feature-window gap (negative = under-capturing)
lane["low_prev"] = (lane["ctr_prev"] < 0.5 * lane["expected_ctr_prev"]).astype(int)
lane["low_last"] = (lane["ctr_last"] < 0.5 * lane["expected_ctr_last"]).astype(int)   # TARGET (proxy)

print("\nTarget column low_last:")
print(lane["low_last"].value_counts().rename({0: "not low", 1: "low"}))
print(f"Base rate (share of pages with low_last = 1): {lane['low_last'].mean():.1%}")

Typical CTR by tier, prev 30 days: {'page_1': 0.34, 'striking': 0.35, 'top_3': 0.57}
Typical CTR by tier, last 30 days: {'page_1': 0.36, 'striking': 0.42, 'top_3': 0.57}

Target column low_last:
low_last
not low    6382
low        4453
Name: count, dtype: int64
Base rate (share of pages with low_last = 1): 41.1%


## 3. Success metric

**Precision@50, judged against the base rate and against the fixed-rule baseline.**

- *Why precision@K:* a reviewer opens a fixed number of pages. What matters is how many of the top 50 are real problems, not overall accuracy.
- *Why K = 50:* it matches the starter pipeline and a plausible sprint size. It is a policy choice and I will test 20 and 100.
- *What "good" means:* a random list of 50 hits at the **base rate**, about 41% here. The fixed rule has to be beaten, and then my model has to beat the fixed rule. A number is only good relative to those two.
- *Also reported:* recall of low-CTR pages and the share of top-50 pages with enough volume to trust (a list full of 100-impression pages would score well and mean little).
- *Validation (week 5+):* hold out whole clients, not random pages, because pages from one client share patterns.

In [ ]:
# The two reference points the metric will be judged against
base_rate = lane["low_last"].mean()

fixed_rule = lane["ctr_prev"] < 0.5          # starter-style fixed cut-off, no position adjustment
flagged = lane[fixed_rule]
print(f"Base rate (random list of pages):                {base_rate:.1%} precision")
print(f"Fixed rule 'CTR < 0.5%' flags {fixed_rule.mean():.0%} of the slice")
print(f"  precision of everything it flags:              {flagged['low_last'].mean():.1%}")
print(f"  lift over base rate:                           {flagged['low_last'].mean() / base_rate:.2f}x")

Base rate (random list of pages):                41.1% precision
Fixed rule 'CTR < 0.5%' flags 79% of the slice
  precision of everything it flags:              48.6%
  lift over base rate:                           1.18x


## 4. The unit of analysis, as a real dataframe

**One row = one page (one `content_id`), with its 90-day and two 30-day search metrics.** It is not a page-day and not a page-query pair. Keeping the grain fixed lets me check for duplicates and group by client for validation.

Columns I keep, and why:
- **Identifiers (grouping only, never features):** `content_id`, `client_id`.
- **Feature-window inputs:** `impressions_prev_30d`, `clicks_prev_30d`, `ctr_prev`, `gap_prev`, `position_tier`, `avg_position`, `main_intent`, `content_type`, `word_count`, `content_age_days`.
- **Target:** `low_last` (from the later window).

**Careful about concentration:** one client holds a large share of the pages, which is why a grouped split matters (shown below).

In [ ]:
cols = ["content_id", "client_id", "position_tier", "avg_position", "impressions_prev_30d",
        "clicks_prev_30d", "ctr_prev", "expected_ctr_prev", "gap_prev", "main_intent", "low_last"]
unit = lane[cols].copy()

print("Shape:", unit.shape)
print("Duplicate content_id:", unit["content_id"].duplicated().sum(), "(should be 0: one row = one page)")
display(unit.head(8).round(2))

print("\nPages per client (concentration check):")
per_client = lane.groupby("client_id").size().sort_values(ascending=False)
print(f"  largest client holds {per_client.iloc[0]:,} pages = {per_client.iloc[0]/len(lane):.0%} of the slice")
print(f"  median client holds {int(per_client.median())} pages")

Shape: (10835, 11)
Duplicate content_id: 0 (should be 0: one row = one page)


,content_id,client_id,position_tier,avg_position,impressions_prev_30d,clicks_prev_30d,ctr_prev,expected_ctr_prev,gap_prev,main_intent,low_last
0,content_304f48230142,client_f369cb89fc,striking,10.6,987,13,1.32,0.35,0.97,transactional,0
3,content_331d6c4de07b,client_19581e27de,page_1,6.2,4206,17,0.40,0.34,0.06,commercial,0
5,content_d4084a4bc775,client_f369cb89fc,page_1,8.5,1009,1,0.10,0.34,-0.24,transactional,1
9,content_c27558df2b0c,client_19581e27de,page_1,4.9,356,0,0.00,0.34,-0.34,informational,1
10,content_d8ee6cc6d642,client_19581e27de,top_3,2.2,6441,117,1.82,0.57,1.25,commercial,0
12,content_42fb2cad9ecf,client_6208ef0f77,page_1,5.6,1357,29,2.14,0.34,1.80,informational,0
16,content_78bd1d4a1d4d,client_6208ef0f77,page_1,8.9,3975,5,0.13,0.34,-0.21,informational,0
17,content_761a44afda12,client_19581e27de,page_1,7.3,3792,1,0.03,0.34,-0.31,transactional,1



Pages per client (concentration check):
  largest client holds 4,212 pages = 39% of the slice
  median client holds 73 pages


## 5. Why ML beats a fixed rule here

**A caution first: ML has to earn its place.** If a position-tier benchmark already ranks pages well, I should say so and stop. The plan is to build that tier baseline first and only keep a model if it clearly beats it on precision@50, on held-out clients.

**Why a fixed cut-off is not enough, from the numbers above:**
- The fixed rule "CTR < 0.5%" flags about **79%** of this slice. A list that long is not a review queue.
- What it flags is only slightly better than picking pages at random: about **49% vs a 41% base rate**.
- Position matters: typical CTR runs about 0.57% in the top 3 against about 0.34% on page 1. One cut-off treats them the same.

**Why there is real signal for a learned method to find:** the position-adjusted gap persists. The next cell shows that a page that was low in the earlier window is much more likely to be low again. So there is pattern here, and the open question is whether combining volume, position, intent, age and content properties predicts persistence better than the simple gap alone.

**Why it is not just "train a model":** the work is in the target definition, the volume floor, the benchmark and the validation. The model is the last and smallest step.

**Link to a real content action:** a high-ranked page goes to a reviewer, who checks the search snippet and intent, then rewrites the title or meta description, or marks it "monitor". I will not claim a rewrite raises CTR. That needs an experiment.

In [ ]:
# Evidence the gap persists (so there is something to learn) and that the fixed rule adds little
persist_if_low = lane.loc[lane["low_prev"] == 1, "low_last"].mean()
persist_if_not = lane.loc[lane["low_prev"] == 0, "low_last"].mean()
print(f"P(low later | low earlier):      {persist_if_low:.1%}")
print(f"P(low later | NOT low earlier):  {persist_if_not:.1%}")
print(f"Position matters: tier benchmark (prev 30d) -> "
      f"{lane.groupby('position_tier')['expected_ctr_prev'].first().round(2).to_dict()}")
print(f"Zero-click pages in the earlier window: {(lane['clicks_prev_30d'] == 0).mean():.0%}"
      "  (ties in the gap score: ranking among them needs volume, a job for the model/baseline)")

P(low later | low earlier):      62.8%
P(low later | NOT low earlier):  22.8%
Position matters: tier benchmark (prev 30d) -> {'page_1': 0.34, 'striking': 0.35, 'top_3': 0.57}
Zero-click pages in the earlier window: 23%  (ties in the gap score: ranking among them needs volume, a job for the model/baseline)
